In [18]:
from sklearn.model_selection import cross_val_score
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score
import joblib

In [2]:
merged_df = pd.read_csv('../features/merged_df.csv')
merged_df = merged_df.dropna()
merged_df = merged_df.iloc[:, 2:]

In [ ]:
y = merged_df['target']
X = merged_df.iloc[:, 2:]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [5]:
rf_model = RandomForestClassifier(
    n_estimators=300,  # number of trees
    max_depth=20,    # no limit on depth
    class_weight='balanced',  # to handle class imbalance
    random_state=42,
    n_jobs=-1
)

In [ ]:
cv_scores = cross_val_score(rf_model, X_scaled, y, cv=10, scoring='accuracy')

print(f" 10-Fold Cross Validation Scores: {cv_scores}")
print(f" Mean Cross Validation Score: {cv_scores.mean()}")

In [29]:
rf_model.fit(X_scaled, y)
y_pred_rf = rf_model.predict(X_scaled)

cm = confusion_matrix(y, y_pred_rf, labels=rf_model.classes_)
print("✅ Confusion Matrix:")
print(cm)

✅ Confusion Matrix:
[[171   1   0   1]
 [  0  18   0   0]
 [  0   0 207   1]
 [  0   0   1 121]]


In [30]:
# Initialize metric lists
sensitivity_list = []
specificity_list = []
mcc_list = []
accuracy_list = []

# Loop over each class
for i in range(len(rf_model.classes_)):
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - (TP + FN + FP)

    # Sensitivity (Recall)
    sensitivity = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0

    # Specificity
    specificity = (TN / (TN + FP)) * 100 if (TN + FP) > 0 else 0

    # MCC
    mcc_numerator = (TP * TN) - (FP * FN)
    mcc_denominator = np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    mcc = (mcc_numerator / mcc_denominator) * 100 if mcc_denominator > 0 else 0

    # Accuracy for the class
    accuracy = ((TP + TN) / (TP + FN + FP + TN)) * 100

    # Store all metrics
    sensitivity_list.append(sensitivity)
    specificity_list.append(specificity)
    mcc_list.append(mcc)
    accuracy_list.append(accuracy)


In [32]:
print("\n📊 Random Forest Classification Report (per class):")
print("=" * 60)
print(f"{'Class':<10}{'Sensitivity':>15}{'Specificity':>15}{'MCC':>10}{'Accuracy':>12}")
print("-" * 60)

for idx, cls in enumerate(rf_model.classes_):
    print(f"{str(cls):<10}{sensitivity_list[idx]:>14.2f}%{specificity_list[idx]:>14.2f}%"
          f"{mcc_list[idx]:>10.2f}%{accuracy_list[idx]:>11.2f}%")

# Step 6: Overall accuracy
overall_accuracy = accuracy_score(y, y_pred_rf) * 100
print("=" * 60)
print(f"✅ Overall RF Accuracy: {overall_accuracy:.2f}%")



📊 Random Forest Classification Report (per class):
Class         Sensitivity    Specificity       MCC    Accuracy
------------------------------------------------------------
Fold1              98.84%        100.00%     99.14%      99.62%
Fold2             100.00%         99.80%     97.24%      99.81%
Fold3              99.52%         99.68%     99.20%      99.62%
Fold4              99.18%         99.50%     98.40%      99.42%
✅ Overall RF Accuracy: 99.23%


In [34]:
average_accuracy = np.mean(accuracy_list)
print(f"\n Average Accuracy (across classes): {average_accuracy:.2f}%")


 Average Accuracy (across classes): 99.62%


In [36]:
joblib.dump(rf_model, '../models/rf_final_model.pkl')
print(" rf model saved as '../models/rf_final_model.pkl'")

 rf model saved as 'rf_final_model.pkl'
